In [0]:
# Databricks notebook source
# ============================================================
# NOTEBOOK: NB_06_Reconciliation
# PURPOSE:  Compares SQL Server baseline with Databricks results.
#
#           This is the CRITICAL acceptance gate.
#           If this notebook shows differences, the migration
#           is NOT ready for production.
#
# PREREQUISITES:
# ------------
# 1. Baseline CSV files loaded into Volume: retailbank_dev/baseline/SQLServer/
# 2. All 5 Databricks notebooks (nb_01 to nb_05) have been run
# 3. Data exists in warehouse.customer_portfolio, etc.
#
# WHAT THIS NOTEBOOK DOES:
# -----------------------
# 1. Loads SQL Server baseline data from CSV files
# 2. Loads Databricks results from warehouse tables
# 3. Compares row counts
# 4. Compares aggregate values (total balance, etc.)
# 5. Compares exceptions by severity
# 6. Compares executive report KPIs
# 7. Finds missing or extra rows
# 8. Produces a PASS/FAIL summary
# ============================================================

# --------------------------------------------------------
# STEP 1: IMPORTS
# --------------------------------------------------------
import pyspark.sql.functions as F
from datetime import datetime
from pyspark.sql.types import DoubleType, IntegerType, StringType

# --------------------------------------------------------
# STEP 2: PARAMETERS
# --------------------------------------------------------
business_date = "2026-01-31"
csv_path = "/Volumes/retailbank_dev/baseline/SQLServer/"

print("=" * 60)
print("RECONCILIATION REPORT")
print("=" * 60)
print(f"Business Date: {business_date}")
print(f"Run Time: {datetime.now()}")
print("")

# --------------------------------------------------------
# STEP 3: FUNCTION TO LOAD CSV WITH PROPER NULL HANDLING
# --------------------------------------------------------
# The CSV files contain the literal string 'NULL' for null values.
# We use the nullValue option to convert 'NULL' to proper NULL.
# We also cast columns to consistent types to avoid Decimal vs Float issues.
# --------------------------------------------------------

def load_csv_with_nulls(file_path, column_names, column_types=None):
    """
    Load a CSV file and replace the string 'NULL' with proper null values.
    
    Parameters:
    -----------
    file_path: str - Path to the CSV file
    column_names: list - List of column names to assign
    column_types: dict - Optional dictionary mapping column names to Spark types
    
    Returns:
    --------
    DataFrame with proper null handling and consistent types
    """
    # Read CSV with null handling
    df = spark.read \
        .option("header", "false") \
        .option("inferSchema", "false") \
        .option("nullValue", "NULL") \
        .option("mode", "DROPMALFORMED") \
        .csv(file_path)
    
    # Assign column names
    for i, col_name in enumerate(column_names):
        df = df.withColumnRenamed(f"_c{i}", col_name)
    
    # Apply column types if provided
    if column_types:
        for col_name, col_type in column_types.items():
            if col_name in df.columns:
                df = df.withColumn(col_name, F.col(col_name).cast(col_type))
    
    return df

# --------------------------------------------------------
# STEP 4: LOAD BASELINE DATA FROM CSV
# --------------------------------------------------------
# We load each CSV file and assign column names.
# For numeric columns, we cast to DoubleType to avoid Decimal vs Float issues.
# --------------------------------------------------------

print("STEP 4: Loading SQL Server baseline data from CSV...")
print("-" * 40)

# 4.1 Load Customer Master baseline
baseline_customer_master = load_csv_with_nulls(
    csv_path + "Baseline_CustomerMaster.csv",
    [
        "customer_id",
        "first_name",
        "last_name",
        "customer_category",
        "branch_code",
        "customer_status",
        "created_date",
        "last_updated_date"
    ]
)

print(f"  CustomerMaster: {baseline_customer_master.count()} rows")

# 4.2 Load Customer Portfolio baseline
baseline_portfolio = load_csv_with_nulls(
    csv_path + "Baseline_CustomerPortfolio.csv",
    [
        "business_date",
        "source_system_code",
        "source_account_number",
        "customer_id",
        "customer_name",
        "customer_category",
        "branch_code",
        "product_code",
        "product_description",
        "product_category",
        "regulatory_category",
        "currency_code",
        "exchange_rate",
        "account_balance",
        "base_currency_balance",
        "account_status",
        "eligible_for_reporting",
        "portfolio_value_band",
        "high_value_customer",
        "product_count"
    ],
    {
        "exchange_rate": DoubleType(),
        "account_balance": DoubleType(),
        "base_currency_balance": DoubleType(),
        "high_value_customer": IntegerType(),
        "product_count": IntegerType()
    }
)

print(f"  CustomerPortfolio: {baseline_portfolio.count()} rows")

# 4.3 Load Portfolio Exceptions baseline
baseline_exceptions = load_csv_with_nulls(
    csv_path + "Baseline_CustomerPortfolioExceptions.csv",
    [
        "business_date",
        "source_system_code",
        "source_account_number",
        "customer_id",
        "rule_code",
        "exception_category",
        "exception_description",
        "exception_value",
        "severity_code",
        "priority_level",
        "sla_hours",
        "escalation_required",
        "escalation_queue",
        "business_area",
        "business_owner",
        "exception_hash",
        "logged_date",
        "resolution_due_date"
    ],
    {
        "priority_level": IntegerType(),
        "sla_hours": IntegerType(),
        "escalation_required": IntegerType()
    }
)

print(f"  Exceptions: {baseline_exceptions.count()} rows")

# 4.4 Load Portfolio Exception Report baseline
baseline_report = load_csv_with_nulls(
    csv_path + "Baseline_PortfolioExceptionReport.csv",
    [
        "business_date",
        "kpi_name",
        "kpi_value",
        "kpi_status"
    ]
)

print(f"  Report: {baseline_report.count()} rows")
print("")
print("  Baseline data loaded successfully.")
print("")

# --------------------------------------------------------
# STEP 5: LOAD DATABRICKS RESULTS
# --------------------------------------------------------
# We read from the warehouse tables that were populated
# by running nb_01 to nb_05.
#
# IMPORTANT: We cast Decimal columns to Double to match the baseline
# data type. This prevents the Decimal vs Float type error.
# --------------------------------------------------------

print("STEP 5: Loading Databricks results...")
print("-" * 40)

# 5.1 Load Customer Portfolio from Databricks
dbx_portfolio = spark.table("retailbank_dev.warehouse.customer_portfolio") \
    .filter(F.col("business_date") == business_date) \
    .withColumn("exchange_rate", F.col("exchange_rate").cast(DoubleType())) \
    .withColumn("account_balance", F.col("account_balance").cast(DoubleType())) \
    .withColumn("base_currency_balance", F.col("base_currency_balance").cast(DoubleType())) \
    .withColumn("high_value_customer", F.col("high_value_customer").cast(IntegerType())) \
    .withColumn("product_count", F.col("product_count").cast(IntegerType()))

print(f"  CustomerPortfolio: {dbx_portfolio.count()} rows")

# 5.2 Load Portfolio Exceptions from Databricks
dbx_exceptions = spark.table("retailbank_dev.warehouse.customer_portfolio_exceptions") \
    .filter(F.col("business_date") == business_date) \
    .withColumn("priority_level", F.col("priority_level").cast(IntegerType())) \
    .withColumn("sla_hours", F.col("sla_hours").cast(IntegerType())) \
    .withColumn("escalation_required", F.col("escalation_required").cast(IntegerType()))

print(f"  Exceptions: {dbx_exceptions.count()} rows")

# 5.3 Load Portfolio Exception Report from Databricks
dbx_report = spark.table("retailbank_dev.reporting.portfolio_exception_report") \
    .filter(F.col("business_date") == business_date)

print(f"  Report: {dbx_report.count()} rows")
print("")
print("  Databricks data loaded successfully.")
print("")

# --------------------------------------------------------
# STEP 6: ROW COUNT COMPARISON
# --------------------------------------------------------
# The simplest check: do the row counts match?
# If not, something is wrong at the fundamental level.
# --------------------------------------------------------

print("STEP 6: Row Count Comparison")
print("-" * 40)

sql_count = baseline_portfolio.count()
dbx_count = dbx_portfolio.count()
row_count_match = (sql_count == dbx_count)
print(f"  CustomerPortfolio: SQL={sql_count}, DBX={dbx_count}, Match={row_count_match}")

sql_count = baseline_exceptions.count()
dbx_count = dbx_exceptions.count()
exc_count_match = (sql_count == dbx_count)
print(f"  Exceptions: SQL={sql_count}, DBX={dbx_count}, Match={exc_count_match}")

sql_count = baseline_report.count()
dbx_count = dbx_report.count()
report_count_match = (sql_count == dbx_count)
print(f"  Report: SQL={sql_count}, DBX={dbx_count}, Match={report_count_match}")

print("")

# --------------------------------------------------------
# STEP 7: AGGREGATE COMPARISON
# --------------------------------------------------------
# We compare the total balance, total accounts, and total
# customers. This is a quick way to catch major differences.
#
# CRITICAL: We use float() to convert Decimal to float
# before performing calculations. This avoids the type error.
# --------------------------------------------------------

print("STEP 7: Aggregate Comparison (CustomerPortfolio)")
print("-" * 40)

# 7.1 SQL Server aggregates
sql_agg = baseline_portfolio.agg(
    F.sum("base_currency_balance").alias("total_balance"),
    F.count("source_account_number").alias("total_accounts"),
    F.countDistinct("customer_id").alias("total_customers")
).collect()[0]

# 7.2 Databricks aggregates
dbx_agg = dbx_portfolio.agg(
    F.sum("base_currency_balance").alias("total_balance"),
    F.count("source_account_number").alias("total_accounts"),
    F.countDistinct("customer_id").alias("total_customers")
).collect()[0]

# Convert to float to avoid Decimal vs Float type error
sql_balance = float(sql_agg.total_balance) if sql_agg.total_balance is not None else 0.0
dbx_balance = float(dbx_agg.total_balance) if dbx_agg.total_balance is not None else 0.0
balance_diff = sql_balance - dbx_balance
balance_match = (abs(balance_diff) < 0.01)

sql_accounts = int(sql_agg.total_accounts) if sql_agg.total_accounts is not None else 0
dbx_accounts = int(dbx_agg.total_accounts) if dbx_agg.total_accounts is not None else 0

sql_customers = int(sql_agg.total_customers) if sql_agg.total_customers is not None else 0
dbx_customers = int(dbx_agg.total_customers) if dbx_agg.total_customers is not None else 0

print(f"  Total Balance SQL: {sql_balance}")
print(f"  Total Balance DBX: {dbx_balance}")
print(f"  Difference: {balance_diff}")
print(f"  Match: {balance_match}")
print("")
print(f"  Total Accounts SQL: {sql_accounts}")
print(f"  Total Accounts DBX: {dbx_accounts}")
print("")
print(f"  Total Customers SQL: {sql_customers}")
print(f"  Total Customers DBX: {dbx_customers}")

print("")

# --------------------------------------------------------
# --------------------------------------------------------
# STEP 8: EXCEPTION COMPARISON
# --------------------------------------------------------
#
# WHAT THIS STEP DOES:
# Compares portfolio exceptions between SQL Server and Databricks.
# Three checks:
#   1. Severity counts — how many CRITICAL, HIGH, etc. on each side?
#   2. Rule code counts — how many DQ001, DQ004, etc. on each side?
#   3. SHA256 hash comparison — the gold standard identity check
#
# BUG THAT WAS FIXED HERE:
# The previous version used row.count which in Python refers to
# the .count() method of the Row object, not the actual value.
# This printed memory addresses like:
#   CRITICAL: <built-in method count of Row object at 0x7f...>
# The fix is to use row["count"] which reads the column value
# by name from the collected Row object.
#
# WHAT WE EXPECT TO SEE:
# SQL Server has 8 exceptions: 3x DQ004 CRITICAL + 5x DQ001 HIGH
# Databricks has 3 exceptions: 3x DQ004 CRITICAL only
#
# The DQ001 gap exists because:
#   SQL Server: anti-joins vs CustomerMaster to find DQ001
#   nb_04:      checks customer_id IS NULL only
#   nb_03 already removed NULL customer_id rows from the portfolio
#   so nb_04 finds zero NULL customer_ids and logs zero DQ001
# --------------------------------------------------------

print("STEP 8: Exception Comparison")
print("-" * 40)

# 8.1 Group SQL Server exceptions by severity
sql_exc = (
    baseline_exceptions
    .groupBy("severity_code")
    .count()
    .orderBy("severity_code")
    .collect()
)

# 8.2 Group Databricks exceptions by severity
dbx_exc = (
    dbx_exceptions
    .groupBy("severity_code")
    .count()
    .orderBy("severity_code")
    .collect()
)

# 8.3 Print severity counts
# IMPORTANT: use row["count"] NOT row.count
# row["count"] reads the aggregated count value by column name.
# row.count is a reference to the Python Row object's own
# .count() method, which prints a memory address, not a number.
print("  SQL Server Exceptions by Severity:")
for row in sql_exc:
    severity = row["severity_code"] if row["severity_code"] else "NULL"
    print(f"    {severity}: {row['count']}")

print("")
print("  Databricks Exceptions by Severity:")
for row in dbx_exc:
    severity = row["severity_code"] if row["severity_code"] else "NULL"
    print(f"    {severity}: {row['count']}")

print("")

# 8.4 Rule code breakdown — shows WHICH rules differ, not just totals
# This is more useful than severity alone because it tells you
# exactly which DQ rule is causing the mismatch.
sql_by_rule = (
    baseline_exceptions
    .groupBy("rule_code", "severity_code")
    .count()
    .orderBy("rule_code")
    .collect()
)
dbx_by_rule = (
    dbx_exceptions
    .groupBy("rule_code", "severity_code")
    .count()
    .orderBy("rule_code")
    .collect()
)

# Build lookup dictionaries: key = (rule_code, severity_code)
sql_rule_map = {(r["rule_code"], r["severity_code"]): r["count"] for r in sql_by_rule}
dbx_rule_map = {(r["rule_code"], r["severity_code"]): r["count"] for r in dbx_by_rule}

all_rule_keys = sorted(set(list(sql_rule_map.keys()) + list(dbx_rule_map.keys())))

print(f"  {'RuleCode':<10} {'Severity':<12} {'SQL':>8} {'DBX':>8} {'Match':>8}")
print(f"  {'-'*10} {'-'*12} {'-'*8} {'-'*8} {'-'*8}")

all_rules_match = True
for key in all_rule_keys:
    rc  = key[0] or "NULL"
    sev = key[1] or "NULL"
    sn  = sql_rule_map.get(key, 0)
    dn  = dbx_rule_map.get(key, 0)
    f   = "✓ MATCH" if sn == dn else "✗ DIFF"
    print(f"  {rc:<10} {sev:<12} {sn:>8} {dn:>8} {f:>8}")
    if sn != dn:
        all_rules_match = False

print("")

# 8.5 SHA256 hash comparison — gold standard identity check
# If the same input data went through the same logic,
# the hash will be identical on both platforms.
# In-SQL-only hashes = exceptions SQL has that Databricks does not.
# In-DBX-only hashes = exceptions Databricks has that SQL does not.
print("  SHA256 hash comparison (exact exception identity):")

sql_hashes = set(
    r["exception_hash"].upper()
    for r in baseline_exceptions.select("exception_hash").collect()
    if r["exception_hash"]
)
dbx_hashes = set(
    r["exception_hash"].upper()
    for r in dbx_exceptions.select("exception_hash").collect()
    if r["exception_hash"]
)

in_both     = sql_hashes & dbx_hashes
in_sql_only = sql_hashes - dbx_hashes
in_dbx_only = dbx_hashes - sql_hashes

print(f"  Hashes on both platforms   : {len(in_both)}")
print(f"  Hashes in SQL Server only  : {len(in_sql_only)}")
print(f"  Hashes in Databricks only  : {len(in_dbx_only)}")

if in_sql_only:
    print("")
    print("  Exceptions in SQL but NOT in Databricks:")
    baseline_exceptions.filter(
        F.upper(F.col("exception_hash")).isin(list(in_sql_only))
    ).select(
        "source_account_number", "customer_id",
        "rule_code", "severity_code"
    ).show(truncate=False)

if in_dbx_only:
    print("")
    print("  Exceptions in Databricks but NOT in SQL:")
    dbx_exceptions.filter(
        F.upper(F.col("exception_hash")).isin(list(in_dbx_only))
    ).select(
        "source_account_number", "customer_id",
        "rule_code", "severity_code"
    ).show(truncate=False)

# 8.6 Overall exception verdict
exc_match = (
    exc_count_match and
    all_rules_match and
    not in_sql_only and
    not in_dbx_only
)

print("")
print(f"  Exceptions Match: {'✓ PASS' if exc_match else '✗ FAIL'}")
print("")

# --------------------------------------------------------
# --------------------------------------------------------
# STEP 9: REPORT COMPARISON
# --------------------------------------------------------
#
# WHAT THIS STEP DOES:
# Compares the four executive KPI rows side by side.
# Produces a clear table showing SQL value, Databricks value,
# and whether they match.
#
# BUG THAT WAS FIXED HERE:
# The previous version compared kpi_value as a raw string.
# This caused two problems:
#   1. "42.11" != "42.11%" even though the numbers are equal.
#      Databricks nb_05 stores values with a % suffix.
#      SQL Server stores them without.
#   2. The comparison assumed both sides ordered rows the same
#      way (sql_report[i] matches dbx_report_rows[i]).
#      If row order differs, this silently compares wrong rows.
# The fix joins by kpi_name and strips % before numeric comparison.
#
# KNOWN DISCREPANCY — SLA Compliance:
# SQL Server baseline shows 0.000000 (RED) — all 3 DQ004 exceptions
# are already past their 2-hour resolution deadline at capture time.
# Databricks nb_05 calculates overdue status using current_timestamp()
# at the moment the notebook runs. If nb_05 runs on a later date,
# the same exceptions may appear on time (not overdue) or vice versa.
# This is a known implementation characteristic, not a code defect.
# It requires formal acceptance in the reconciliation evidence pack.
#
# KNOWN DISCREPANCY — Exception Rate:
# SQL Server: 42.11% = 8 exceptions / 19 accounts
# Databricks: 15.79% = 3 exceptions / 19 accounts
# This flows directly from the DQ001 gap in Step 8.
# Resolving DQ001 in nb_04 will also fix this KPI.
# --------------------------------------------------------

print("STEP 9: Report Comparison")
print("-" * 40)

def clean_kpi_value(v):
    """
    Strips the % suffix and whitespace from a KPI value string.
    This allows numeric comparison regardless of whether
    SQL Server or Databricks included the % character.

    Examples:
        "42.11%"   -> "42.11"
        "42.11"    -> "42.11"
        "0.000000" -> "0.000000"
        "POOR"     -> "POOR"
    """
    return str(v or "").replace("%", "").strip()

def kpi_values_match(sql_val, dbx_val):
    """
    Compares two KPI values intelligently.
    Tries numeric comparison first with 0.1 tolerance —
    this handles cases like "42.11" vs "42.11%" or
    "0.000000" vs "0.0" which are the same number.
    Falls back to case-insensitive string comparison
    for text KPIs like "POOR" or "Normal Operations".
    """
    try:
        return abs(float(clean_kpi_value(sql_val)) -
                   float(clean_kpi_value(dbx_val))) < 0.1
    except (ValueError, TypeError):
        return clean_kpi_value(sql_val).upper() == clean_kpi_value(dbx_val).upper()

# Build dictionaries keyed by kpi_name so we join by name,
# not by row position. This is robust regardless of row order.
sql_kpi_dict = {
    row["kpi_name"]: {"value": row["kpi_value"], "status": row["kpi_status"]}
    for row in baseline_report.select("kpi_name", "kpi_value", "kpi_status").collect()
}
dbx_kpi_dict = {
    row["kpi_name"]: {"value": row["kpi_value"], "status": row["kpi_status"]}
    for row in dbx_report.select("kpi_name", "kpi_value", "kpi_status").collect()
}

# Print side-by-side comparison table
print(f"  {'KPI':<25} {'SQL Value':>12} {'DBX Value':>12} "
      f"{'SQL RAG':>8} {'DBX RAG':>8} {'Match':>8}")
print(f"  {'-'*25} {'-'*12} {'-'*12} {'-'*8} {'-'*8} {'-'*8}")

report_match  = True
sla_diff_found = False
rate_diff_found = False

all_kpi_names = sorted(set(list(sql_kpi_dict.keys()) + list(dbx_kpi_dict.keys())))

for kpi in all_kpi_names:
    sql_info = sql_kpi_dict.get(kpi, {"value": "MISSING", "status": "MISSING"})
    dbx_info = dbx_kpi_dict.get(kpi, {"value": "MISSING", "status": "MISSING"})

    sql_val = str(sql_info["value"] or "").strip()
    dbx_val = str(dbx_info["value"] or "").strip()
    sql_rag = str(sql_info["status"] or "").strip()
    dbx_rag = str(dbx_info["status"] or "").strip()

    val_match  = kpi_values_match(sql_val, dbx_val)
    rag_match  = (sql_rag.upper() == dbx_rag.upper())
    full_match = val_match and rag_match
    flag       = "✓ MATCH" if full_match else "✗ DIFF"

    print(f"  {kpi:<25} {sql_val:>12} {dbx_val:>12} "
          f"{sql_rag:>8} {dbx_rag:>8} {flag:>8}")

    if not full_match:
        report_match = False
        if "SLA" in kpi.upper():
            sla_diff_found = True
        if "Rate" in kpi or "RATE" in kpi.upper():
            rate_diff_found = True

print("")

# Print known-difference notes for any KPIs that failed
# so the reader understands the cause without further investigation
if sla_diff_found:
    print("  NOTE — SLA Compliance difference:")
    print("  SQL Server baseline captured SLA status at a fixed point in time.")
    print("  nb_05 recalculates overdue status using current_timestamp() at")
    print("  runtime. The same exception can appear overdue or on-time")
    print("  depending on when nb_05 runs vs the resolution_due_date.")
    print("  This is a known implementation characteristic, not a code defect.")
    print("  Formal programme acceptance required.")
    print("")

if rate_diff_found:
    print("  NOTE — Exception Rate difference:")
    print("  SQL Server: 42.11% = 8 exceptions / 19 accounts")
    print("  Databricks: 15.79% = 3 exceptions / 19 accounts")
    print("  Root cause: DQ001 gap in nb_04 (see Step 8 detail above).")
    print("  Resolving DQ001 will fix this KPI automatically.")
    print("")

print(f"  Report Match: {'✓ PASS' if report_match else '✗ FAIL'}")
print("")

# --------------------------------------------------------
# STEP 10: FIND DISCREPANCIES
# --------------------------------------------------------
# We find rows that exist in SQL Server but are missing
# in Databricks, and vice versa.
# --------------------------------------------------------

print("STEP 10: Finding discrepancies")
print("-" * 40)

# 10.1 Rows in SQL Server but missing in Databricks
missing_in_db = baseline_portfolio.alias("sql").join(
    dbx_portfolio.alias("dbx"),
    F.col("sql.source_account_number") == F.col("dbx.source_account_number"),
    "left_anti"
)

missing_count = missing_in_db.count()
if missing_count > 0:
    print(f"  Rows in SQL Server but missing in Databricks: {missing_count}")
    print("  First 5 missing rows:")
    missing_in_db.select("source_account_number", "customer_id", "base_currency_balance").show(5, truncate=False)
else:
    print("  No missing rows found.")

# 10.2 Rows in Databricks but missing in SQL Server
extra_in_db = dbx_portfolio.alias("dbx").join(
    baseline_portfolio.alias("sql"),
    F.col("dbx.source_account_number") == F.col("sql.source_account_number"),
    "left_anti"
)

extra_count = extra_in_db.count()
if extra_count > 0:
    print(f"  Rows in Databricks but missing in SQL Server: {extra_count}")
    print("  First 5 extra rows:")
    extra_in_db.select("source_account_number", "customer_id", "base_currency_balance").show(5, truncate=False)
else:
    print("  No extra rows found.")

print("")

# --------------------------------------------------------
# STEP 11: FINAL STATUS
# --------------------------------------------------------
# We combine all checks into a single PASS/FAIL status.
# All checks must pass for the migration to be accepted.
# --------------------------------------------------------

print("=" * 60)
print("RECONCILIATION SUMMARY")
print("=" * 60)

all_pass = (
    row_count_match and
    balance_match and
    exc_match and
    report_match and
    missing_count == 0 and
    extra_count == 0
)

print(f"  Row Count Match: {'PASS' if row_count_match else 'FAIL'}")
print(f"  Balance Match: {'PASS' if balance_match else 'FAIL'}")
print(f"  Exceptions Match: {'PASS' if exc_match else 'FAIL'}")
print(f"  Report Match: {'PASS' if report_match else 'FAIL'}")
print(f"  No Missing Rows: {'PASS' if missing_count == 0 else 'FAIL'}")
print(f"  No Extra Rows: {'PASS' if extra_count == 0 else 'FAIL'}")
print("")

if all_pass:
    print("  OVERALL STATUS: PASS - READY FOR SIGN-OFF")
else:
    print("  OVERALL STATUS: FAIL - INVESTIGATE DISCREPANCIES")

print("=" * 60)

# --------------------------------------------------------
# STEP 12: DETAILED DISCREPANCY REPORT (If FAIL)
# --------------------------------------------------------
# If there are failures, we provide more detail to help
# with investigation.
# --------------------------------------------------------

if not all_pass:
    print("")
    print("DETAILED DISCREPANCY REPORT")
    print("-" * 40)
    
    if not row_count_match:
        print("  Row count mismatch: Investigate why counts differ.")
        print(f"    SQL: {baseline_portfolio.count()}, DBX: {dbx_portfolio.count()}")
    
    if not balance_match:
        print("  Balance mismatch: Check currency conversion or rounding.")
        print(f"    SQL: {sql_balance}, DBX: {dbx_balance}")
    
    if not exc_match:
        print("  Exception mismatch: Check DQ rules in nb_04.")
        print("    Compare severity counts above.")
    
    if not report_match:
        print("  Report mismatch: Check KPI calculations in nb_05.")
        print("    Compare KPI values above.")
    
    if missing_count > 0:
        print("  Missing rows: Check nb_03 extraction logic.")
        print(f"    {missing_count} rows missing in Databricks.")
    
    if extra_count > 0:
        print("  Extra rows: Check if Databricks has additional data.")
        print(f"    {extra_count} extra rows in Databricks.")

print("")
print("RECONCILIATION COMPLETE.")

RECONCILIATION REPORT
Business Date: 2026-01-31
Run Time: 2026-08-24 03:29:00.312725

STEP 4: Loading SQL Server baseline data from CSV...
----------------------------------------
  CustomerMaster: 19 rows
  CustomerPortfolio: 19 rows
  Exceptions: 8 rows
  Report: 4 rows

  Baseline data loaded successfully.

STEP 5: Loading Databricks results...
----------------------------------------
  CustomerPortfolio: 19 rows
  Exceptions: 6 rows
  Report: 4 rows

  Databricks data loaded successfully.

STEP 6: Row Count Comparison
----------------------------------------
  CustomerPortfolio: SQL=19, DBX=19, Match=True
  Exceptions: SQL=8, DBX=6, Match=False
  Report: SQL=4, DBX=4, Match=True

STEP 7: Aggregate Comparison (CustomerPortfolio)
----------------------------------------
  Total Balance SQL: 3261330.0
  Total Balance DBX: 3261330.0
  Difference: 0.0
  Match: True

  Total Accounts SQL: 19
  Total Accounts DBX: 19

  Total Customers SQL: 12
  Total Customers DBX: 12

STEP 8: Exception 